In [ ]:
import os
import json
import shutil
import random
import numpy as np
from pathlib import Path
from collections import Counter
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}, GPUs: {torch.cuda.device_count()}")

## 1. Find Datasets

In [ ]:
# Find dataset paths
input_path = Path("/kaggle/input")

def find_train_dir(base):
    """Recursively find train folder with class subdirs"""
    for root, dirs, files in os.walk(base):
        root = Path(root)
        if root.name.lower() == 'train':
            subdirs = [d for d in root.iterdir() if d.is_dir()]
            if subdirs and list(subdirs[0].glob('*.jpg')) + list(subdirs[0].glob('*.JPG')):
                return root
    return None

# Auto-detect
DS1_TRAIN = None
DS2_TRAIN = None

for item in input_path.iterdir():
    name = item.name.lower()
    if 'new-plant-diseases' in name or 'new_plant_diseases' in name:
        DS1_TRAIN = find_train_dir(item)
    elif 'plant-disease' in name or 'plant_disease' in name:
        if 'new' not in name:
            DS2_TRAIN = find_train_dir(item) or item

print(f"Dataset 1 (vipoooool): {DS1_TRAIN}")
print(f"Dataset 2 (Bangladesh): {DS2_TRAIN}")

if not DS1_TRAIN or not DS2_TRAIN:
    print("\n❌ ERROR: Datasets not found! Add them via + Add Input")
else:
    print("\n✓ Both datasets found!")

## 2. Collect Images with STRICT Limits

In [ ]:
# CLASS LIMITS - Key to fixing Cotton bias!
COTTON_LIMIT = 300      # Strict limit for Cotton
DEFAULT_LIMIT = 1000    # Normal limit for other classes
MIN_IMAGES = 50         # Minimum to include class

def get_crop_name(class_name):
    """Extract crop from class name"""
    # Handle Crop___Disease format
    if '___' in class_name:
        return class_name.split('___')[0]
    # Handle Disease_(Crop) format
    if '(' in class_name and ')' in class_name:
        start = class_name.rfind('(')
        end = class_name.rfind(')')
        return class_name[start+1:end]
    return class_name

def normalize_to_unified(class_name):
    """Convert any format to Crop___Disease"""
    class_name = class_name.strip()
    
    # Already unified
    if '___' in class_name:
        return class_name
    
    # Disease_(Crop) -> Crop___Disease
    if '(' in class_name and ')' in class_name:
        start = class_name.rfind('(')
        end = class_name.rfind(')')
        crop = class_name[start+1:end]
        disease = class_name[:start].rstrip('_')
        return f"{crop}___{disease}"
    
    return f"Unknown___{class_name}"

# Test
print("Normalization test:")
tests = ["Apple___Black_rot", "Black_rot_(Apple)", "Healthy_(Cotton)", "Cotton___healthy"]
for t in tests:
    print(f"  {t} -> {normalize_to_unified(t)}")

In [ ]:
def collect_images(dataset_path, source_name):
    """Collect all images from dataset"""
    images = {}  # unified_class -> list of image paths
    
    if dataset_path is None:
        return images
    
    for class_dir in dataset_path.iterdir():
        if not class_dir.is_dir():
            continue
        
        unified = normalize_to_unified(class_dir.name)
        
        # Get images
        class_images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.JPG')) + \
                       list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg'))
        
        if unified not in images:
            images[unified] = []
        images[unified].extend(class_images)
    
    print(f"{source_name}: {len(images)} classes, {sum(len(v) for v in images.values())} images")
    return images

# Collect from both
ds1_images = collect_images(DS1_TRAIN, "Dataset 1")
ds2_images = collect_images(DS2_TRAIN, "Dataset 2")

# Merge
all_images = {}
for unified, paths in ds1_images.items():
    if unified not in all_images:
        all_images[unified] = []
    all_images[unified].extend(paths)

for unified, paths in ds2_images.items():
    if unified not in all_images:
        all_images[unified] = []
    all_images[unified].extend(paths)

print(f"\nMerged: {len(all_images)} unique classes")

In [ ]:
# Show Cotton classes BEFORE limiting
print("=" * 50)
print("COTTON CLASSES (before limiting):")
print("=" * 50)
cotton_total = 0
for unified, paths in sorted(all_images.items()):
    crop = get_crop_name(unified)
    if 'cotton' in crop.lower():
        print(f"  {unified}: {len(paths)} images")
        cotton_total += len(paths)
print(f"\nTotal Cotton images: {cotton_total}")
print(f"This is why model predicts Cotton for everything!")

In [ ]:
# Apply STRICT limits
balanced_images = {}
skipped = []

for unified, paths in all_images.items():
    # Skip if too few
    if len(paths) < MIN_IMAGES:
        skipped.append((unified, len(paths)))
        continue
    
    # Determine limit based on crop
    crop = get_crop_name(unified).lower()
    if 'cotton' in crop:
        limit = COTTON_LIMIT  # Strict limit!
    else:
        limit = DEFAULT_LIMIT
    
    # Shuffle and limit
    random.shuffle(paths)
    balanced_images[unified] = paths[:limit]

print(f"After balancing: {len(balanced_images)} classes")
print(f"Skipped {len(skipped)} classes (too few images)")

# Show Cotton AFTER limiting
print("\n" + "=" * 50)
print("COTTON CLASSES (after limiting):")
print("=" * 50)
cotton_total = 0
for unified, paths in sorted(balanced_images.items()):
    crop = get_crop_name(unified)
    if 'cotton' in crop.lower():
        print(f"  {unified}: {len(paths)} images")
        cotton_total += len(paths)
print(f"\nTotal Cotton images: {cotton_total} (was much higher!)")

## 3. Create Train/Valid Split

In [ ]:
TRAIN_RATIO = 0.85
MERGED = Path("/kaggle/working/merged_balanced")

# Clean
if MERGED.exists():
    shutil.rmtree(MERGED)

(MERGED / "train").mkdir(parents=True)
(MERGED / "valid").mkdir(parents=True)

# Copy with split
final_classes = sorted(balanced_images.keys())

for unified in tqdm(final_classes, desc="Creating dataset"):
    paths = balanced_images[unified]
    random.shuffle(paths)
    
    split = int(len(paths) * TRAIN_RATIO)
    train_imgs = paths[:split]
    valid_imgs = paths[split:]
    
    # Create dirs
    train_dir = MERGED / "train" / unified
    valid_dir = MERGED / "valid" / unified
    train_dir.mkdir(exist_ok=True)
    valid_dir.mkdir(exist_ok=True)
    
    # Copy
    for i, p in enumerate(train_imgs):
        shutil.copy2(p, train_dir / f"{i:04d}.jpg")
    for i, p in enumerate(valid_imgs):
        shutil.copy2(p, valid_dir / f"{i:04d}.jpg")

# Save class names
with open("/kaggle/working/class_names.json", "w") as f:
    json.dump(final_classes, f, indent=2)

NUM_CLASSES = len(final_classes)
print(f"\n✓ Created dataset with {NUM_CLASSES} classes")
print(f"  Saved to: /kaggle/working/class_names.json")

In [ ]:
# Show class distribution
print("\n" + "=" * 50)
print("FINAL CLASS DISTRIBUTION BY CROP")
print("=" * 50)

crop_stats = Counter()
crop_images = Counter()

for unified, paths in balanced_images.items():
    crop = get_crop_name(unified)
    crop_stats[crop] += 1
    crop_images[crop] += len(paths)

for crop in sorted(crop_stats.keys()):
    print(f"{crop:30} {crop_stats[crop]:3} classes, {crop_images[crop]:5} images")

print(f"\nTotal: {sum(crop_stats.values())} classes, {sum(crop_images.values())} images")

## 4. Create DataLoaders

In [ ]:
class PlantDataset(Dataset):
    def __init__(self, root, class_names, transform=None):
        self.root = Path(root)
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.transform = transform
        self.samples = []
        
        for cls in class_names:
            cls_dir = self.root / cls
            if cls_dir.exists():
                for img in cls_dir.glob('*'):
                    if img.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                        self.samples.append((img, self.class_to_idx[cls]))
        
        random.shuffle(self.samples)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, label
        except:
            return self.__getitem__(random.randint(0, len(self)-1))

# Transforms
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(30),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

valid_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = PlantDataset(MERGED / "train", final_classes, train_tf)
valid_ds = PlantDataset(MERGED / "valid", final_classes, valid_tf)

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=4, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_ds)} images, Valid: {len(valid_ds)} images")
print(f"Batches: Train={len(train_loader)}, Valid={len(valid_loader)}")

## 5. Focal Loss (Reduces Cotton Bias)

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss - reduces the loss for well-classified examples,
    focusing training on hard, misclassified examples.
    This helps prevent the model from always predicting Cotton.
    """
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction='none')
    
    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)  # probability of correct class
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

print("✓ Focal Loss defined (gamma=2)")
print("  This penalizes confident wrong predictions like 'Cotton for everything'")

## 6. Build Model

In [ ]:
class DiseaseModel(nn.Module):
    def __init__(self, num_classes, dropout=0.5):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_feat = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_feat, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

model = DiseaseModel(NUM_CLASSES, dropout=0.5)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"Using {torch.cuda.device_count()} GPUs")

model = model.to(device)
print(f"Model ready: {NUM_CLASSES} classes")

## 7. Training

In [ ]:
EPOCHS = 25
LR = 0.001
PATIENCE = 6

criterion = FocalLoss(gamma=2)  # Key: Focal loss!
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f"Epochs: {EPOCHS}")
print(f"Loss: Focal Loss (gamma=2) - prevents Cotton bias")
print(f"Optimizer: AdamW (weight_decay=0.01)")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for imgs, labels in tqdm(loader, desc="Train"):
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, pred = out.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
    
    return total_loss / len(loader), 100 * correct / total

def validate(model, loader, criterion, class_names):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    predictions = Counter()
    
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Valid"):
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = criterion(out, labels)
            
            total_loss += loss.item()
            _, pred = out.max(1)
            total += labels.size(0)
            correct += pred.eq(labels).sum().item()
            
            # Track prediction distribution
            for p in pred.cpu().numpy():
                predictions[class_names[p]] += 1
    
    return total_loss / len(loader), 100 * correct / total, predictions

In [ ]:
# Training loop
best_acc = 0
patience_cnt = 0
history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}

print("\n" + "="*60)
print("TRAINING - Watching for Cotton Bias")
print("="*60 + "\n")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 40)
    
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    valid_loss, valid_acc, preds = validate(model, valid_loader, criterion, final_classes)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['valid_loss'].append(valid_loss)
    history['valid_acc'].append(valid_acc)
    
    print(f"Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
    print(f"Valid: Loss={valid_loss:.4f}, Acc={valid_acc:.2f}%")
    
    # Check for Cotton bias
    print("\nTop 5 predictions (checking for bias):")
    for cls, cnt in preds.most_common(5):
        pct = 100 * cnt / sum(preds.values())
        flag = "⚠️ BIAS!" if pct > 20 else ""
        print(f"  {cls}: {pct:.1f}% {flag}")
    
    # Save best
    if valid_acc > best_acc:
        best_acc = valid_acc
        patience_cnt = 0
        
        state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save({
            'model_state_dict': state,
            'class_names': final_classes,
            'num_classes': NUM_CLASSES,
            'valid_acc': valid_acc,
            'epoch': epoch + 1
        }, '/kaggle/working/disease_model_best.pth')
        print(f"\n✓ Best model saved! Acc={valid_acc:.2f}%")
    else:
        patience_cnt += 1
        print(f"\nNo improvement ({patience_cnt}/{PATIENCE})")
    
    if patience_cnt >= PATIENCE:
        print(f"\n⚠️ Early stopping at epoch {epoch+1}")
        break

print(f"\n" + "="*60)
print(f"DONE! Best accuracy: {best_acc:.2f}%")
print("="*60)

In [ ]:
# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['valid_loss'], label='Valid')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['valid_acc'], label='Valid')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_plot.png', dpi=150)
plt.show()

## 8. Test Predictions

In [ ]:
# Load best and test
ckpt = torch.load('/kaggle/working/disease_model_best.pth')
test_model = DiseaseModel(NUM_CLASSES)
test_model.load_state_dict(ckpt['model_state_dict'])
test_model = test_model.to(device).eval()

print(f"Loaded model: {ckpt['num_classes']} classes, {ckpt['valid_acc']:.2f}% acc")

# Test on samples
crops_to_test = ['Apple', 'Grape', 'Tomato', 'Rice', 'Mango', 'Potato']
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for idx, crop in enumerate(crops_to_test):
    ax = axes[idx // 3, idx % 3]
    
    # Find a class for this crop
    crop_classes = [c for c in final_classes if c.startswith(crop)]
    if not crop_classes:
        ax.set_title(f"{crop}: Not found")
        ax.axis('off')
        continue
    
    cls = random.choice(crop_classes)
    cls_dir = MERGED / "valid" / cls
    imgs = list(cls_dir.glob('*'))
    
    if not imgs:
        ax.set_title(f"{crop}: No images")
        ax.axis('off')
        continue
    
    img_path = random.choice(imgs)
    img = Image.open(img_path).convert('RGB')
    
    # Predict
    with torch.no_grad():
        inp = valid_tf(img).unsqueeze(0).to(device)
        out = test_model(inp)
        probs = torch.softmax(out, 1)[0]
        top_prob, top_idx = probs.max(0)
        pred_class = final_classes[top_idx]
    
    # Show
    ax.imshow(img)
    correct = "✓" if pred_class == cls else "✗"
    ax.set_title(f"True: {cls}\nPred: {pred_class}\n{top_prob:.1%} {correct}")
    ax.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/test_predictions.png', dpi=150)
plt.show()

## 9. Download Files

Download from `/kaggle/working/`:
1. `disease_model_best.pth` → rename to `disease_model_pytorch.pth`
2. `class_names.json`

Place both in `backend/models/` folder

In [ ]:
print("\n" + "="*60)
print("FILES TO DOWNLOAD")
print("="*60)

files = [
    '/kaggle/working/disease_model_best.pth',
    '/kaggle/working/class_names.json',
    '/kaggle/working/training_plot.png',
    '/kaggle/working/test_predictions.png'
]

for f in files:
    p = Path(f)
    if p.exists():
        size = p.stat().st_size / (1024*1024)
        print(f"✓ {p.name} ({size:.1f} MB)")

print("\n" + "="*60)
print("INSTRUCTIONS:")
print("="*60)
print("1. Click folder icon 📁 on the right")
print("2. Go to /kaggle/working/")
print("3. Right-click → Download:")
print("   - disease_model_best.pth")
print("   - class_names.json")
print("4. Put in backend/models/")
print("5. Rename disease_model_best.pth → disease_model_pytorch.pth")